In [1]:
# CELL 0: Set Run Parameters
import os

os.environ["CACHE_LOOKBACK"] = "41"
os.environ["CACHE_START_DATE"] = "1998-01-01"

In [2]:
# CELL 1
import time

start_time = time.time()

In [3]:
# CELL 2: Setup
import sys, os
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

Mounted at /content/drive


In [4]:
# CELL 3: Load Data & Initialize Cache
from core.settings import TradingConfig, CacheConfig
from core.paths import OUTPUT_DIR
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from data_pipeline.screener import UniverseScreener
from data_pipeline.cache import CheckpointAlphaCache

print("Loading preprocessed data...")
data = load_processed_data()
df_ohlcv = data.df_ohlcv
macro_df = data.macro_df
features_df = data.features_df

# Tripwire validation on processed feature schemas
expected_macro_cols = [
    "Macro_Trend",
    "Macro_Vix_Z",
    "Macro_Vix_Ratio",
    "Mkt_Vol_63d_Z",
]

expected_feature_cols = [
    "Mom_252_21",
    "ResMom_126",
    "Range_Pos_52w",
    "IR_63",
    "RSI",
    "DD_21",
    "Trend_R2_63",
    "Beta_Down_63",
    "ATRP",
    "ER_63",
    "Mom_126",
    "IVol_63",
]

assert all(
    col in macro_df.columns for col in expected_macro_cols
), f"macro_df missing expected engineered columns! Found: {macro_df.columns.tolist()}"

assert all(
    col in features_df.columns for col in expected_feature_cols
), f"features_df missing expected columns! Found: {features_df.columns.tolist()}"

print("✅ Data validation passed: Schemas and contracts verified.")

config = TradingConfig()

# Reconstruct wide df_close dynamically and sync calendar
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

NOTEBOOKS_RLVR_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2
PROJECT_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks
GLOBAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data
GLOBAL_PROCESSED_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data/processed
LOCAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data
OUTPUT_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output

Loading preprocessed data...
✅ Data validation passed: Schemas and contracts verified.


In [ ]:
import pandas as pd

# Display all columns without truncation
pd.set_option("display.max_columns", None)

# Increase maximum width of the output display (in characters)
pd.set_option("display.width", 2000)

# Expand column width so text inside cells isn't truncated
pd.set_option("display.max_colwidth", None)

In [5]:
print("--- macro_df columns ---")
print(macro_df.columns.tolist())
print("\n--- macro_df sample ---")
print(macro_df.tail(2))

print("\n--- features_df columns ---")
print(features_df.columns.tolist())

--- macro_df columns ---
['Mkt_Ret', 'Mkt_Ret_Z', 'Macro_Trend', 'Macro_Trend_Z', 'Yield_Curve_10Y2Y_Z', 'Macro_Trend_Vel_Z', 'Macro_Trend_Mom', 'Macro_Vix_Z', 'Macro_Vix_Ratio', 'Mkt_Vol_63d_Z']

--- macro_df sample ---
             Mkt_Ret  Mkt_Ret_Z  Macro_Trend  Macro_Trend_Z  \
Date                                                          
2026-09-03  0.010468   1.151856     0.090018       0.212462   
2026-09-04 -0.003854  -0.571064     0.084976       0.059501   

            Yield_Curve_10Y2Y_Z  Macro_Trend_Vel_Z  Macro_Trend_Mom  \
Date                                                                  
2026-09-03            -0.929766          -0.867652        -0.867652   
2026-09-04            -1.115408          -1.055424        -1.055424   

            Macro_Vix_Z  Macro_Vix_Ratio  Mkt_Vol_63d_Z  
Date                                                     
2026-09-03    -1.346174         0.822044       0.707033  
2026-09-04    -1.225449         0.825099       0.152430  

--- feat

In [6]:
# CELL 4: Build Cache
import os
from pathlib import Path
from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR  # Added LOCAL_DATA_DIR

# Create directories if they do not exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Successfully ensured output directory exists: {OUTPUT_DIR}")
print(f"Successfully ensured local data directory exists: {LOCAL_DATA_DIR}")

Successfully ensured output directory exists: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output
Successfully ensured local data directory exists: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data


In [7]:
# CELL 5: Build Cache
cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()

# Add a warning if the cache file already exists
if cache_file_path.exists():
    print(
        f"Warning: Cache file already exists at {cache_file_path}. Resuming build process."
    )

cache = CheckpointAlphaCache(
    screener=screener, config=config, lookbacks=[CacheConfig.LOOKBACK]
)

cache.build(
    start_date=CacheConfig.START_DATE,
    end_date=None,
    checkpoint_path=cache_file_path,
    save_every_n_days=2000,  # save every ~7 minutes
)

# print(cache.describe())

[INFO] Building AlphaCache with automatic resumption support:
  - Start Date:               1998-01-01
  - End Date Limit:           None (Processing all available dates)
  - Total target dates:       7213
  - Already completed:        0
  - Remaining to compute:     7213
  - Checkpoint save interval: Every 2000 processed days
  - Output location:          /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data/alpha_cache_41d_1998.parquet
  Processed 20/7213 remaining days... (est. 54:48 till completion)
  Processed 40/7213 remaining days... (est. 54:31 till completion)
  Processed 60/7213 remaining days... (est. 54:06 till completion)
  Processed 80/7213 remaining days... (est. 53:46 till completion)
  Processed 100/7213 remaining days... (est. 53:37 till completion)
  Processed 120/7213 remaining days... (est. 53:31 till completion)
  Processed 140/7213 remaining days... (est. 53:44 till completion)
  Processed 160/7213 remaining days... (est

In [8]:
print(f"File saved to: {cache_file_path}")
print(f"File exists: {cache_file_path.exists()}")

File saved to: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data/alpha_cache_41d_1998.parquet
File exists: True


In [9]:
# CELL 6: Timing
end_time = time.time()
print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}")

Time: 00:51:32


In [10]:
# CELL 7: Disconnect from TPU
from google.colab import runtime

# This will restart and unassign the current runtime,
# effectively disconnecting from the T4 GPU.
runtime.unassign()
print("Runtime unassigned. This session has been disconnected.")

Runtime unassigned. This session has been disconnected.
